# Multimodal data loading with `grain`

This tutorial focuses on **one thing**: how to feed data into `stix`. Training,
sampling and the model internals are covered in the
[training-and-sampling tutorial](./1.training_and_sampling.ipynb). Here
we only build the data pipeline that produces `Batch` objects.

We use [`grain`](https://google-grain.readthedocs.io/en/latest/index.html), a
JAX-friendly data-loading library. Beyond ordinary loading, `grain` gives us
**reproducibility** and **checkpointing** of the dataset state, so an
interrupted run can resume the data stream exactly where it stopped.

Whatever the data, plugging it into `stix` comes down to producing
[`Batch`](https://instadeepai.github.io/stix/api_reference/typing/index.html#stix.typing.data.Batch)
objects — and there are **two ways** to get there, both shown here:

1. **Raw samples + a `dict_to_batch` map.** Use this when the source is
   **external** (a HuggingFace dataset) or a **reusable primitive** you feed into
   several pipelines — you can't, or don't want to, bake the `Batch` into it.
   `grain` fetches `source[i]` and a `.map(dict_to_batch)` step wraps each
   (batched) dict into a `Batch`.
2. **A source that yields `Batch` objects directly.** When you own a simple,
   single-purpose source, it's cleaner to build the `Batch` inside it and drop
   the conversion step — this is what the training-and-sampling tutorial does.

Either way the source is **map-style** (`__getitem__(index)` + `__len__()`): a
HuggingFace split already is one, and for synthetic data a function or a tiny
class suffices. `grain` owns the pipeline in between:
`source → shuffle → repeat → batch → (map)`.

## Structure of the notebook

1. **A real dataset** — MNIST from HuggingFace (raw dicts + `dict_to_batch`).
2. **A minimal custom source** — an inline four-corner GMM sampler that returns a `Batch` directly.
3. **A richer custom source** — the `GMMRingGenerator`, a reusable primitive kept dict-returning + `dict_to_batch`.
4. **Checkpointing the stream** — saving and restoring the data state.


## 0. Installation

We recommend running this notebook in a **fresh virtual environment**. 

Copy the notebook into some new directory. Then, from a terminal, in the new directory containing the notebook (`2.grain_multimodal_dataloading.ipynb`):
```
python -m venv my_env
source my_env/bin/activate
pip install notebook ipykernel

python -m ipykernel install --user --name my_env --display-name "my_env"

jupyter notebook
```

The next cell installs `stix` from PyPI together with the extra plotting package this notebook uses.

In [ ]:
%pip install stix-ml matplotlib

## Imports

In [ ]:
from functools import partial
from typing import Any

import grain
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from datasets import load_dataset
from matplotlib.colors import BoundaryNorm

# The container types every stix component consumes.
from stix.typing import Batch, RawSourceTargetPair

## 1. A real dataset: MNIST from HuggingFace

We start from a dataset that is already a map-style source: a HuggingFace
`datasets` split supports `dataset[i]` and `len(dataset)` out of the box, so it
plugs straight into `grain` with no wrapper.


In [ ]:
hf_dataset = load_dataset("ylecun/mnist")

hf_train: Any = hf_dataset["train"]
hf_test: Any = hf_dataset["test"]

print(f"Train split : {len(hf_train)} samples")
print(f"One element : {hf_train[0].keys()}")  # {'image': <PIL.Image>, 'label': int}

### The two conversion functions

MNIST is an **external** source: `datasets` hands us a fixed dictionary per
element, so we can't make it emit a `Batch` — this is the `dict_to_batch` case
(way 1). `grain` lets us attach per-element transforms via `.map(...)`; we use
two:

- **`preprocess_mnist`** : an **optional** data-shaping step. The raw MNIST
  `image` is a PIL image of `uint8` values in `[0, 255]`; the model expects a
  float array in `[-1, 1]` with an explicit channel axis. This function is where
  any such per-modality reshaping/rescaling lives — add, remove or extend it to
  suit your data.

- **`dict_to_batch`** : the step that wraps the raw dictionary into
  a [`Batch`](https://instadeepai.github.io/stix/api_reference/typing/index.html#stix.typing.data.Batch).
  Each modality becomes a `RawSourceTargetPair(target=..., source=...)` inside a
  plain per-modality dict. We set `source=None`: in the *one-sided* setting the
  source is implicit Gaussian noise, so only the `target` (the real data) is
  provided. The remaining `Batch` fields (`context_data`, masks, ...) default to
  `None` and are omitted here.

In [ ]:
def preprocess_mnist(sample: dict) -> dict:
    """Optional shaping: PIL uint8 image -> float32 array in [-1, 1] with a channel axis.

    Stays `numpy`: the arrays only cross to the device at the end of the pipeline.
    """
    img = np.asarray(sample["image"], dtype=np.float32)  # (..., 28, 28), range [0, 255]
    img = img / 127.5 - 1.0  # -> [-1, 1]
    img = img[..., None]  # -> (..., 28, 28, 1)
    return {"image": img, "label": np.asarray(sample["label"], dtype=np.int32)}


def dict_to_batch(sample: dict) -> Batch:
    """Wrap the raw dict into a stix Batch (one RawSourceTargetPair per modality)."""
    raw_batch = {
        "image": RawSourceTargetPair(target=sample["image"], source=None),
        "label": RawSourceTargetPair(target=sample["label"], source=None),
    }
    return Batch(
        raw_batch=raw_batch,
        is_discrete={"image": False, "label": True},
    )

### Wiring the pipeline

We chain the `grain` operations. The order matters — `.map(...)` steps come **after** `.batch(...)`, so both transforms operate on an already batched dictionary: one call per batch rather than one per sample.

- **`seed`** : fixes the pipeline's PRNG for reproducibility;
- **`shuffle`** : shuffles the elements;
- **`repeat`** : turns the finite split into an infinite stream (the step-based
  training loop pulls batches indefinitely);
- **`batch`** : groups elements; `drop_remainder=True` keeps a static shape;
- **`map`** : applies `preprocess_mnist` then `dict_to_batch`;
- **`to_iter_dataset`** : hands back the Python iterator we pull batches from.
- **`device_put`** : moves each finished batch to the device, with prefetching.

This is the order grain's own [JAX tutorial](https://github.com/google/grain/blob/main/docs/tutorials/jax_training_tutorial.md) recommends: random-access transforms first, iterator conversion last.

Two rules of thumb this ordering encodes, worth copying into your own pipelines:

1. **grain runs on the CPU; JAX makes the device arrays.** For an *external* source (like MNIST), keep the elements `numpy` all the way through the pipeline and move them across in one go at the end — that is what the `grain.experimental.device_put` wrapper below does (`jax.device_put` in a final `.map(...)` would work too). Host-side elements are what let grain parallelise loading across processes with [`mp_prefetch`](https://google-grain.readthedocs.io/en/latest/index.html), which device arrays cannot cross.
2. **`to_iter_dataset` goes last.** Everything above it supports random access, which is what lets `batch` group elements by slicing rather than pulling them through an iterator one at a time.

> **For a *synthetic* source the first rule inverts.** When you write the draw yourself there is nothing to load from disk, so the cheapest thing is to let JAX build the batch directly: grain shuffles and groups the **indices**, and one `jax.jit`-compiled call turns a batch of indices into a batch of samples. Sections 2 and 3 below do exactly that, and need no `device_put` — the sampler already returns device arrays.


In [ ]:
seed = 0
batch_size = 256

# `device_put` takes (and returns) an IterDataset, so it wraps the finished
# pipeline: everything above it stays host-side `numpy`.
train_dataset = grain.experimental.device_put(
    grain.MapDataset.source(hf_train)
    .seed(seed)
    .shuffle()
    .repeat()
    .batch(batch_size, drop_remainder=True)
    .map(preprocess_mnist)
    .map(dict_to_batch)
    .to_iter_dataset(),
    device=jax.devices()[0],
)

train_iter = iter(train_dataset)

### Inspecting and visualising a batch

`next(train_iter)` yields a fully-formed `Batch`. We read each modality's
`target` and display a grid of raw training images together with
their labels.


In [ ]:
batch = next(train_iter)
images = batch.raw_batch["image"].target  # (batch_size, 28, 28, 1) in [-1, 1]
labels = batch.raw_batch["label"].target  # (batch_size,)

print(f"Image : {images.shape}")
print(f"Label : {labels.shape}")

# Rescale [-1, 1] -> [0, 1] for display.
images_view = (images + 1.0) / 2.0

fig, axes = plt.subplots(4, 8, figsize=(10, 5))
for ax, img, lbl in zip(axes.flat, images_view, labels):
    ax.imshow(jnp.squeeze(img, axis=-1), cmap="gray")
    ax.set_title(int(lbl), fontsize=9)
    ax.axis("off")
fig.suptitle("MNIST batch (raw training data)")
plt.tight_layout()
plt.show()

## 2. A minimal custom source: an inline GMM sampler

When the data is synthetic there is no HuggingFace split to lean on, so we
provide the source ourselves. It doesn't need to be a class: `grain` can take a
plain `range` of indices and `.map(...)` a function that turns each index into a
sample. This is exactly the four-corner Gaussian mixture used in the
[training and sampling tutorial](./1.training_and_sampling.ipynb) — one Gaussian
per corner of a square, each draw returning the `"coordinates"` of the point and
the `"indices"` of the mode it came from.

Because we own this source and it maps one-to-one onto a one-sided `Batch`, we
take use the 2nd approach: `sample_gmm` builds the `Batch` itself, so the pipeline needs no
`dict_to_batch` map step at all (contrast with MNIST above).

Because the draw is ours to write, we vectorise it: `grain` shuffles and groups
the **indices**, and one `jax.jit`-compiled call turns a batch of indices into a
batch of samples.

Two details make a *synthetic* source behave well under `grain`:

- **`jax.random.fold_in(key, index)`** derives a fresh key *from the index*, so
  element `i` is deterministic and independent of every other element: the same
  `i` always yields the same sample (reproducibility), and shuffling only changes
  the order indices are visited, not their content.
- **`MapDataset.range(int(1e9))`** is just a big index stream; `.repeat()` makes
  it effectively infinite, so the length only needs to exceed a single pass.

`sample_gmm` takes the batch of indices `grain` hands it and returns one batched
`Batch` directly, so the pipeline is the same as the MNIST one with `sample_gmm`
in place of `preprocess_mnist` + `dict_to_batch`. Note `jax.vmap` doing the work:
the maths inside `draw_one` is still written for a single sample, which is usually
the clearest way to read it.

In [ ]:
CORNERS = jnp.array(
    [[-1.0, -1.0], [1.0, -1.0], [-1.0, 1.0], [1.0, 1.0]], dtype=jnp.float32
)


@jax.jit  # Note: one compiled call builds the whole batch! See the pipeline below
def sample_gmm(batch_indices: jax.Array, key: jax.Array) -> Batch:
    """Draw one GMM sample per index in ``batch_indices`` and wrap them in a one-sided ``Batch``.

    Pick a corner, add Gaussian noise, and return each modality as a
    ``RawSourceTargetPair`` with ``source=None`` (the source is implicit Gaussian noise).
    """

    def draw_one(sample_key: jax.Array) -> tuple[jax.Array, jax.Array]:
        """One sample: the maths stays per-sample, ``vmap`` turns it into a batch."""
        idx_key, noise_key = jax.random.split(sample_key)
        idx = jax.random.randint(idx_key, (), 0, len(CORNERS))
        coordinates = (
            jax.random.normal(noise_key, (2,), dtype=jnp.float32) * 0.2 + CORNERS[idx]
        )
        return coordinates, idx

    keys = jax.vmap(partial(jax.random.fold_in, key))(batch_indices)
    coordinates, idx = jax.vmap(draw_one)(keys)
    raw_batch = {
        "coordinates": RawSourceTargetPair(target=coordinates, source=None),
        "indices": RawSourceTargetPair(target=idx.astype(jnp.int32), source=None),
    }
    return Batch(
        raw_batch=raw_batch,
        is_discrete={"coordinates": False, "indices": True},
    )


train_dataset = (
    grain.MapDataset.range(int(1e9))
    .seed(seed)
    .shuffle()
    .repeat()
    .batch(batch_size, drop_remainder=True)  # groups indices
    .map(partial(sample_gmm, key=jax.random.key(0)))  # indices -> batched Batch
    .to_iter_dataset()
)

train_iter = iter(train_dataset)

A small helper to scatter 2D points coloured by their categorical mode —
we reuse it for the ring in section 3.


In [ ]:
def scatter_by_mode(coords, modes, num_modes, title):
    """Scatter 2D points coloured by a discrete mode index."""
    cmap = plt.get_cmap("tab10", num_modes)
    norm = BoundaryNorm([i - 0.5 for i in range(num_modes + 1)], num_modes)

    fig, ax = plt.subplots(figsize=(6, 6))
    sc = ax.scatter(
        coords[:, 0],
        coords[:, 1],
        s=10,
        c=modes,
        cmap=cmap,
        norm=norm,
        alpha=0.6,
    )
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal")
    fig.colorbar(sc, ax=ax, label="mode index", ticks=range(num_modes))
    plt.tight_layout()
    plt.show()


batch = next(train_iter)
coords = batch.raw_batch["coordinates"].target  # (batch_size, 2)
indices = batch.raw_batch["indices"].target  # (batch_size,)

print(f"Coordinates : {coords.shape}")
print(f"Indices     : {indices.shape}")

scatter_by_mode(coords, indices, num_modes=4, title="GMM: four-corner mixture")

## 3. A richer custom source: `GMMRingGenerator`

Real experiments often need more structured synthetic data. When a source has
state to precompute (here: mode centers on a ring, per-mode weights), a small
**class** is the natural fit — it builds the geometry once in `__init__` and
exposes the same grain contract (`__len__` + `__getitem__`) so it drops into the
pipeline just like the function above.

`GMMRingGenerator` places `num_modes` equally-spaced Gaussians on a ring and
supports an **unequal mixture**: an optional `weights` argument gives a per-mode
selection probability (defaulting to uniform), turned into categorical logits and
sampled with `jax.random.categorical`. It yields the same
`{"coordinates", "indices"}` modalities as the section-2 sampler.

Unlike that sampler, it deliberately stays **dict-returning** (way 1). Keeping the source raw — and wrapping it with a small `dict_to_batch` (exactly as we did for MNIST) — lets one generator assemble *different* `Batch` layouts (one-sided, two-sided, or with extra conditioning fields) without baking any single one into the source. We define it inline here.

In [ ]:
class GMMRingGenerator:
    """Equiprobable (or weighted) Gaussians equally spaced on a ring.

    Builds the ring geometry once in ``__init__``; ``sample_batch`` turns a batch of
    indices into a batch of ``{"coordinates", "indices"}`` samples.
    """

    def __init__(self, num_modes=8, radius=2.0, sigma=0.05, weights=None, seed=0):
        """Precompute the ring geometry and per-mode categorical logits."""
        self.key = jax.random.key(seed)

        self.num_modes = num_modes
        self.sigma = sigma

        weights = (
            jnp.ones(num_modes) / num_modes if weights is None else jnp.asarray(weights)
        )
        self.logits = jnp.log(weights)  # categorical logits over modes

        angles = jnp.linspace(0.0, 2 * jnp.pi, num_modes, endpoint=False)
        self.centers = radius * jnp.stack([jnp.cos(angles), jnp.sin(angles)], axis=1)

    def sample_batch(self, batch_indices: jax.Array) -> dict:
        """Return one sample per index in ``batch_indices`` (coordinates + mode index)."""

        def draw_one(sample_key: jax.Array) -> dict:
            """One sample: the maths stays per-sample, ``vmap`` turns it into a batch."""
            idx_key, noise_key = jax.random.split(sample_key)
            idx = jax.random.categorical(idx_key, self.logits)
            noise = jax.random.normal(noise_key, (2,)) * self.sigma
            return {
                "indices": idx.astype(jnp.int32),
                "coordinates": self.centers[idx] + noise,
            }

        keys = jax.vmap(partial(jax.random.fold_in, self.key))(batch_indices)
        return jax.vmap(draw_one)(keys)

In [ ]:
NUM_MODES = 8

# Unequal mixture: probabilities ramp from low to high around the ring, then
# normalise to sum to 1. The most likely mode is ~4x as frequent as the rarest.
weights = jnp.linspace(1.0, 4.0, NUM_MODES)
weights = weights / weights.sum()
print("Mode probabilities:", [f"{w:.2f}" for w in weights])


def dict_to_batch(sample: dict) -> Batch:
    """Wrap a GMM-ring sample (coordinates + indices) into a one-sided Batch."""
    raw_batch = {
        "coordinates": RawSourceTargetPair(target=sample["coordinates"], source=None),
        "indices": RawSourceTargetPair(target=sample["indices"], source=None),
    }
    return Batch(
        raw_batch=raw_batch,
        is_discrete={"coordinates": False, "indices": True},
    )


train_source = GMMRingGenerator(
    num_modes=NUM_MODES, radius=2.0, sigma=0.1, weights=weights, seed=0
)
val_source = GMMRingGenerator(
    num_modes=NUM_MODES, radius=2.0, sigma=0.1, weights=weights, seed=42
)

train_dataset = (
    grain.MapDataset.range(int(1e9))
    .seed(seed)
    .shuffle()
    .repeat()
    .batch(batch_size, drop_remainder=True)  # groups indices
    # jit the bound method: the precomputed ring is captured as constants.
    .map(jax.jit(train_source.sample_batch))  # indices -> dict of batched arrays
    .map(dict_to_batch)  # raw dict -> Batch (the ring stays dict-returning)
    .to_iter_dataset()
)

train_iter = iter(train_dataset)

In [ ]:
batch = next(train_iter)
coords = batch.raw_batch["coordinates"].target
indices = batch.raw_batch["indices"].target

print(f"Coordinates : {coords.shape}")
print(f"Indices     : {indices.shape}")

scatter_by_mode(
    coords,
    indices,
    num_modes=NUM_MODES,
    title=f"GMMRingGenerator: {NUM_MODES}-mode ring (unequal weights)",
)

## 4. Saving and restoring the data state

A headline feature of `grain` is that the **position in the data stream is
checkpointable**. `stix`'s training loop relies on this to resume an interrupted
run exactly where it stopped (same shuffle order, same next element) so a
resumed run is bit-identical to one that never stopped.

The iterator returned by `iter(dataset)` exposes two methods:

- **`get_state()`** → a small, JSON-serializable dict describing the cursor
  (here just `{"next_index": ...}`);
- **`set_state(state)`** → moves the iterator back (or forward) to a saved cursor.

Below we snapshot the state, write it to disk, consume the next batch, then
restore the snapshot and confirm we get the **same** batch back.


In [ ]:
import json

# Consume a couple of batches, then snapshot the data state (the stream cursor).
next(train_iter)
next(train_iter)
state = train_iter.get_state()
print("Saved state:", state)

# The state is plain JSON, so it can be written to disk as part of a checkpoint ...
with open("data_state.json", "w") as f:
    json.dump(state, f)
# ... and read back later (e.g. when resuming a run).
with open("data_state.json") as f:
    restored_state = json.load(f)

# Pull the next batch from the live iterator, then rewind to the saved cursor
# and pull again — the two batches should be identical.
batch_after_save = next(train_iter)

train_iter.set_state(restored_state)
batch_after_restore = next(train_iter)

same = bool(
    jnp.all(
        batch_after_save.raw_batch["indices"].target
        == batch_after_restore.raw_batch["indices"].target
    )
)
print("Batch after restore matches the original:", same)

### Takeaways

Across three very different sources — a HuggingFace dataset, a four-corner
mixture, and a configurable ring — the goal never changed: expose a **map-style
source** (`__getitem__` + `__len__`) and get each element into a `Batch`. Only
*where* the `Batch` is built differed:

- **External or reusable sources** (MNIST, `GMMRingGenerator`) stay raw and add a
  `.map(dict_to_batch)` step — you can't change the source, or it feeds several
  pipelines that each assemble a different `Batch`.
- **Bespoke, single-purpose sources** (the inline `sample_gmm`) build the `Batch`
  themselves and skip the conversion entirely.

Everything else (`shuffle → repeat → batch → (map)`) is boilerplate `grain`
handles for you. To bring your own data, pick whichever of the two fits and reuse
the rest.
